# This notebook adds differential arrival times to the data.

There are two common ways to generate differential arrival times:

### 1. Generate them from absolute arrival times. For event si observed at stations rj and rk, the differential time can be obtained by subtracting two absolute arrivals:

dt_i,jk = t_i,j - t_i,k

This approach does not require waveform data, but its uncertainty is usually larger. This notebook uses this method.

### 2. Generate them from waveform cross-correlation.

See:

VanDecar, J. C., & Crosson, R. S. (1990). Determination of teleseismic relative phase arrival times using multi-channel cross-correlation and least squares. Bulletin of the Seismological Society of America, 80(1), 150-169.

Cross-correlation can provide smaller errors, but it requires waveform data and careful choices of time windows and selection thresholds. Cross-correlation differential times are reliable only when the waveforms are sufficiently similar, usually with high correlation coefficients.

In [ ]:
# Import TomoATT data-processing utilities
import sys
sys.path.append('../utils')
import functions_for_data as ffd

# 1. Read the Arrival-Time Data File

In [ ]:
# Read data
fname = "output_data/step3_src_rec_filtered.dat"
[ev_info_obs, st_info_obs] = ffd.read_src_rec_file(fname)

# Data plot (optional); set fname = None to skip saving the figure.
ffd.fig_ev_st_distribution_dep(ev_info_obs, st_info_obs, fname = None)

# 2. Generate Common-Source Differential Arrival Times

These data describe differential arrival times from the same earthquake recorded by nearby stations.

Common-source differential arrival times are insensitive to source-location uncertainty and origin-time errors, which helps improve imaging reliability. They are useful for tomography.

Two thresholds are used here: station spacing smaller than 100 km and an angle smaller than 30 degrees between the two great-circle paths.

These thresholds ensure that the two paths overlap as much as possible near the source, so the common-source differential data can reduce the impact of source uncertainty on imaging results.

The constraints can be relaxed, but data that do not satisfy them have weak path overlap near the source and behave more like two independent absolute arrival-time constraints.

In [ ]:
# Station spacing smaller than 100 km and great-circle path angle smaller than 30 degrees
dis_thd = 100 # distance between two stations should be less than 100 km
azi_thd = 30  # the angle bwteen two great circle paths from the common source to two separated receivers should be less than 30

ev_info = ffd.generate_cs_dif(ev_info_obs,st_info_obs,dis_thd,azi_thd)

# 3. Optional: Generate Common-Receiver Differential Arrival Times

These data describe differential arrival times from nearby earthquakes recorded by the same station.

Common-receiver differential arrival times are insensitive to structural uncertainty near the station and can improve relative earthquake relocation. They are useful for relocation-focused studies.

Two thresholds are used here: event spacing smaller than 3 km and an angle smaller than 5 degrees between the two great-circle paths.

These thresholds ensure that the two paths overlap as much as possible, so the differential data mainly constrain relative source locations and structure between sources.

These thresholds are stricter than those for common-source differential data because the number of earthquakes can be large, and relaxed thresholds may create an extremely large data set.

However, when source uncertainty is large, common-receiver differential arrival times can amplify the influence of source uncertainty and may introduce imaging errors.

In [ ]:
# Because the ISC source uncertainties in the Turkey region are relatively large, common-receiver differential times are not used here.
# For high-precision relocation, consider adding common-receiver differential times.

# Event spacing smaller than 3 km and great-circle path angle smaller than 5 degrees
# dis_thd = 3  # distance between two earthquakes should be less than 3 km
# azi_thd = 5  # the angle bwteen two great circle paths from the common receiver to two separated sources should be less than 5 degree

# ev_info = ffd.generate_cr_dif(ev_info,st_info_obs,dis_thd,azi_thd)

# 4. Output the Processed Data

In [ ]:
# Write data to the target directory
import os

# Specify the data directory
out_path = "output_data"
os.makedirs(out_path,exist_ok=True)

# Save as a TomoATT-format data file
out_fname = "%s/step4_src_rec_cs.dat"%(out_path)
ffd.write_src_rec_file(out_fname,ev_info,st_info_obs)

# 5. Use This Data Set for the Tomography Workflow

In [ ]:
# Write data to the target directory
import os

# Specify the data directory
out_path = "../1_src_rec_files"
os.makedirs(out_path,exist_ok=True)

# Save as a TomoATT-format data file
out_fname = "%s/src_rec_file.dat"%(out_path)
ffd.write_src_rec_file(out_fname,ev_info,st_info_obs)